In [22]:
# Analysis: Compare heart rate vs BMI as predictors of calories burned
import numpy as np
import pandas as pd

CSV = '/Users/mel-neiquaholloway/Downloads/gym_members_exercise_tracking_synthetic_data.csv'

# Load dataset
df = pd.read_csv(CSV)
# Clean column names and rename useful columns
df.columns = [c.strip() for c in df.columns]
cols = {
    'Avg_BPM': 'avg_bpm',
    'Max_BPM': 'max_bpm',
    'Resting_BPM': 'rest_bpm',
    'Calories_Burned': 'calories',
    'BMI': 'bmi',
    'Session_Duration (hours)': 'duration_hrs'
}
for k,v in cols.items():
    if k in df.columns:
        df = df.rename(columns={k:v})

# Coerce numerics
for col in ['avg_bpm','max_bpm','rest_bpm','calories','bmi','duration_hrs','Weight (kg)','Age']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Build preferred heart-rate column (avg -> max -> rest)
df['hr_preferred'] = np.nan
if 'avg_bpm' in df.columns:
    df['hr_preferred'] = df['avg_bpm']
if 'max_bpm' in df.columns:
    df['hr_preferred'] = df['hr_preferred'].fillna(df['max_bpm'])
if 'rest_bpm' in df.columns:
    df['hr_preferred'] = df['hr_preferred'].fillna(df['rest_bpm'])

# Filter rows with required fields
mask = (~df['calories'].isna()) & (~df['hr_preferred'].isna()) & (~df['bmi'].isna()) & (~df['duration_hrs'].isna())
df_model = df[mask].copy()
print(f'Rows total: {len(df)}, usable rows for model (calories+HR+bmi+duration): {len(df_model)}')

# Pairwise Pearson correlations helper
def pearson(a, b):
    ia = ~a.isna()
    ib = ~b.isna()
    m = ia & ib
    if m.sum() < 2:
        return np.nan
    return float(np.corrcoef(a[m], b[m])[0,1])

print('\nPairwise Pearson correlations with calories:')
for name in ['avg_bpm','max_bpm','rest_bpm','hr_preferred','bmi','Weight (kg)']:
    if name in df.columns or name == 'hr_preferred':
        r = pearson(df_model.get(name, pd.Series(dtype=float)), df_model['calories'])
        print(f'  {name}: r={r:.3f}')

# Standardized linear regression utilities
def standardize(arr):
    arr = np.array(arr, dtype=float)
    m = np.nanmean(arr)
    s = np.nanstd(arr, ddof=0)
    if s == 0 or np.isnan(s):
        return arr - m
    return (arr - m) / s

def fit_standardized(X_df, y_series):
    X = np.array(X_df, dtype=float)
    y = np.array(y_series, dtype=float)
    mask = np.isfinite(y) & np.all(np.isfinite(X), axis=1)
    X = X[mask]
    y = y[mask]
    if len(y) == 0:
        return None
    Xs = np.vstack([standardize(X[:,i]) for i in range(X.shape[1])]).T
    ys = standardize(y)
    coef, *_ = np.linalg.lstsq(Xs, ys, rcond=None)
    yhat = Xs.dot(coef)
    ss_res = ((ys - yhat)**2).sum()
    ss_tot = ((ys - ys.mean())**2).sum()
    r2 = 1 - ss_res/ss_tot if ss_tot>0 else np.nan
    return {'coef': coef, 'r2': r2, 'n': len(y)}

# Prepare predictors and run models
base = df_model[['hr_preferred','bmi','duration_hrs','Age']].copy()
if 'Age' not in base.columns:
    base['Age'] = np.nan
results = {}
results['HR alone'] = fit_standardized(base[['hr_preferred']], df_model['calories'])
results['BMI alone'] = fit_standardized(base[['bmi']], df_model['calories'])
results['HR + BMI'] = fit_standardized(base[['hr_preferred','bmi']], df_model['calories'])
results['Full (HR,BMI,duration,Age)'] = fit_standardized(base[['hr_preferred','bmi','duration_hrs','Age']], df_model['calories'])

print('\nRegression results (standardized coefficients and R^2):')
for k, v in results.items():
    if v is None:
        print(f'  {k}: insufficient data')
    else:
        # determine names for printing coefs
        if 'Full' in k:
            names = ['hr_preferred','bmi','duration_hrs','Age']
        elif 'HR alone' in k:
            names = ['hr_preferred']
        elif 'BMI alone' in k:
            names = ['bmi']
        else:
            names = ['hr_preferred','bmi']
        coefs = ', '.join([f'{n}:{c:.3f}' for n,c in zip(names, v['coef'])])
        print(f'  {k}: n={v['n']}, R2={v['r2']:.3f}, coefs=({coefs})')

# Partial R^2 drops from full model
full = results.get('Full (HR,BMI,duration,Age)')
if full is not None:
    without_hr = fit_standardized(base[['bmi','duration_hrs','Age']], df_model['calories'])
    without_bmi = fit_standardized(base[['hr_preferred','duration_hrs','Age']], df_model['calories'])
    print('\nPartial R^2 drops from Full model:')
    print(f'  Full R2 = {full['r2']:.3f}')
    if without_hr: print(f'  Without HR: R2={without_hr['r2']:.3f}, drop={full['r2'] - without_hr['r2']:.3f}')
    if without_bmi: print(f'  Without BMI: R2={without_bmi['r2']:.3f}, drop={full['r2'] - without_bmi['r2']:.3f}')

print('\nNote: coefficients shown are from regression on standardized variables (so magnitudes are comparable). No p-values computed.')


Rows total: 1800, usable rows for model (calories+HR+bmi+duration): 1725

Pairwise Pearson correlations with calories:
  avg_bpm: r=0.042
  max_bpm: r=-0.017
  rest_bpm: r=-0.113
  hr_preferred: r=0.026
  bmi: r=-0.040
  Weight (kg): r=-0.011

Regression results (standardized coefficients and R^2):
  HR alone: n=1725, R2=0.001, coefs=(hr_preferred:0.026)
  BMI alone: n=1725, R2=0.002, coefs=(bmi:-0.040)
  HR + BMI: n=1725, R2=0.002, coefs=(hr_preferred:0.026, bmi:-0.040)
  Full (HR,BMI,duration,Age): n=1715, R2=0.003, coefs=(hr_preferred:0.024, bmi:-0.043, duration_hrs:0.026, Age:-0.012)

Partial R^2 drops from Full model:
  Full R2 = 0.003
  Without HR: R2=0.003, drop=0.001
  Without BMI: R2=0.001, drop=0.002

Note: coefficients shown are from regression on standardized variables (so magnitudes are comparable). No p-values computed.
